# Parallel GPU Computing (PGC) - Lab Evaluation 2
## Theme 7: CUDA Warp Divergence Analysis

**Team Members:**
- Adarsh Walad (01fe24bci)
- Nilaksha Nath (01fe24bci087)
- Bachu venkata siva jaideep (01fe24bci084)
- Prathamesh Eedigi (01fe24bci006)

---
### Aim
Show how branch divergence inside a warp slows a CUDA kernel and learn architectural techniques (warp alignment, data sorting, and branch-free predication) to mitigate its cost.

### Step 1: Check GPU and Toolchain Information
Verify the presence of NVIDIA GPU, Driver version, and CUDA Compiler (NVCC).

In [ ]:
!nvidia-smi
!nvcc --version

### Step 2: Source Code Implementation
We implement four distinct kernel designs to evaluate SIMT warp execution behavior:
1. **Divergent Kernel**: Branches based on `threadIdx.x % 2`. Alternating adjacent threads take different paths, forcing 50% lane masking and branch serialization.
2. **Warp-Aligned Kernel**: Branches based on `((threadIdx.x / 32) % 2)`. Condition is uniform across each 32-thread warp, ensuring 100% active lane mask.
3. **Data-Dependent Kernel**: Branches based on runtime input values `input[i] > threshold`.
4. **Branch-Free Kernel**: Predicated execution calculating both candidate paths and selecting with ternary operator (`selp`).

In [ ]:
with open('../src/warp_divergence.cu', 'r') as f:
    cuda_code = f.read()
print(f"CUDA source loaded successfully ({len(cuda_code.splitlines())} lines).")

### Step 3: Compile and Execute
Compile with target compute architecture `sm_89` (NVIDIA Ada Lovelace):

In [ ]:
!nvcc -O2 -std=c++14 -arch=sm_89 ../src/warp_divergence.cu -o ../bin/warp_divergence
!../bin/warp_divergence

### Step 4: Visualizing Performance Metrics and Active Masks

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../results/timing_data.csv')
display(df)

plt.figure(figsize=(8, 4.5))
plt.bar(df['kernel_name'], df['avg_time_ms'], color=['#e74c3c', '#2ecc71', '#3498db', '#9b59b6'])
plt.title('CUDA Kernel Execution Time (NVIDIA RTX 4500 Ada - 2^24 elements)')
plt.ylabel('Execution Time (ms)')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()